# 🌀 Kuantum Kredi Kartı Dolandırıcılık Tespiti (QSVM)

Bu notebook, **SSB Kuantum Algoritma Yarışması** kapsamında geliştirilen, kredi kartı harcama verilerindeki sahtekarlık (dolandırıcılık) olaylarını klasik makine öğrenmesi modelleri ve **Kuantum Destek Vektör Makineleri (QSVM)** ile tespit etmeyi amaçlayan eğitim ve test adımlarını içerir.

### 📋 Notebook Akışı:
1. Kütüphanelerin Yüklenmesi
2. Veri Ön İşleme (Sınıf Dengeleme ve PCA Boyut Azaltma)
3. Klasik Makine Öğrenmesi Baseline Modelleri (SVM, RF, GBM)
4. Kuantum SVM Modelinin Oluşturulması ve Eğitilmesi (Qiskit v2.5.0)
5. Sonuçların Grafiksel Karşılaştırılması

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Qiskit imports
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

import warnings
warnings.filterwarnings("ignore")

print("Tüm kütüphaneler başarıyla yüklendi!")

## 1. Veri Ön İşleme (Data Preprocessing)

Kaggle'dan indirdiğimiz kredi kartı veri seti oldukça büyüktür ve yüksek derecede dengesizdir (%0.17 sahtekarlık oranı). Bu aşamada:
1. Veri setinden dengeli bir alt küme (80 normal, 80 sahte işlem) oluşturacağız.
2. Boyutları 6 bileşene (PCA) indirgeyeceğiz.
3. Eğitim ve test setlerini (%80 / %20) oluşturup numpy formatında kaydedeceğiz.

In [ ]:
# Veri ön işlemeyi çalıştıralım (6 PCA bileşeni, her sınıftan 80 örnek)
import sys
sys.path.append("..")
from src.data_preprocessing import preprocess_data

preprocess_data(n_components=6, sample_size_per_class=80, test_size=0.2, random_state=42)

# Numpy dosyalarını yükleyelim (Notebook'un nerede çalıştırıldığına göre dinamik klasör seçimi)
processed_dir = os.path.join("data", "processed")
if not os.path.exists(processed_dir):
    processed_dir = os.path.join("..", "data", "processed")

X_train = np.load(os.path.join(processed_dir, "train_x.npy"))
X_test = np.load(os.path.join(processed_dir, "test_x.npy"))
y_train = np.load(os.path.join(processed_dir, "train_y.npy"))
y_test = np.load(os.path.join(processed_dir, "test_y.npy"))

print("\nEğitim veri kümesi şekli:", X_train.shape)
print("Test veri kümesi şekli:", X_test.shape)
print("Eğitim Sınıf Dağılımı:", np.bincount(y_train))
print("Test Sınıf Dağılımı:", np.bincount(y_test))

## 2. Klasik Makine Öğrenmesi Baseline Modelleri

Kuantum SVM modelinin performansını karşılaştırmak amacıyla klasik makine öğrenmesi modellerini eğiteceğiz:
1. Support Vector Machine (RBF kernel)
2. Random Forest Classifier
3. Gradient Boosting Classifier

In [ ]:
# Klasik modelleri eğitip metriklerini hesaplayalım
models = {
    "Support Vector Machine (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting (GBM)": GradientBoostingClassifier(random_state=42)
}

classical_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    classical_results[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    }
    
    print(f"{name} - F1-Score: {f1:.4f}, ROC-AUC: {auc:.4f}, Accuracy: {acc:.4f}")

## 3. Kuantum Destek Vektör Makinesi (QSVM)

Kuantum SVM, klasik verileri `ZZFeatureMap` kullanarak kuantum özellik uzayına kodlar. 
Ardından benzerlik matrisi (`FidelityQuantumKernel`) kuantum simülatörü (`StatevectorSampler`) ile hesaplanarak klasik SVM modeline precomputed kernel olarak beslenir.

In [ ]:
# 1. Kuantum Özellik Haritasını (ZZFeatureMap) tanımlayalım
num_features = X_train.shape[1]
feature_map = ZZFeatureMap(feature_dimension=num_features, reps=2, entanglement='linear')

# 2. Sampler ve Fidelity araçlarını ilklendirelim
sampler = StatevectorSampler()
fidelity = ComputeUncompute(sampler=sampler)

# 3. Kuantum Çekirdeği oluşturalım
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

# 4. Eğitim ve Test Çekirdek Matrislerini hesaplayalım
print("Eğitim Kuantum Kernel Matrisi hesaplanıyor (128x128)...")
train_matrix = quantum_kernel.evaluate(x_vec=X_train)
print("Test Kuantum Kernel Matrisi hesaplanıyor (32x128)...")
test_matrix = quantum_kernel.evaluate(x_vec=X_test, y_vec=X_train)

# 5. Precomputed SVM modelini eğitelim
print("\nQSVM modeli eğitiliyor...")
qsvm = SVC(kernel='precomputed', probability=True, random_state=42)
qsvm.fit(train_matrix, y_train)

# 6. Tahmin ve Değerlendirme
y_pred_q = qsvm.predict(test_matrix)
y_prob_q = qsvm.predict_proba(test_matrix)[:, 1]

acc_q = accuracy_score(y_test, y_pred_q)
prec_q = precision_score(y_test, y_pred_q)
rec_q = recall_score(y_test, y_pred_q)
f1_q = f1_score(y_test, y_pred_q)
auc_q = roc_auc_score(y_test, y_prob_q)

print("\nKuantum SVM (QSVM) Sonuçları:")
print(f"  F1-Score: {f1_q:.4f}, ROC-AUC: {auc_q:.4f}, Accuracy: {acc_q:.4f}")

## 4. Sonuçların Görselleştirilmesi

Elde edilen klasik baseline ve QSVM performans sonuçlarını karşılaştırmak amacıyla bir bar grafik çizdireceğiz ve Kuantum Kernel matrisini görselleştireceğiz.

In [ ]:
# Performans karşılaştırma grafiğini çizelim
models_list = ["SVM (RBF)", "Random Forest", "Gradient Boosting", "Quantum SVM (QSVM)"]
f1_scores = [classical_results["Support Vector Machine (RBF)"]["F1-Score"], 
             classical_results["Random Forest"]["F1-Score"], 
             classical_results["Gradient Boosting"]["F1-Score"], 
             f1_q]
auc_scores = [classical_results["Support Vector Machine (RBF)"]["ROC-AUC"], 
              classical_results["Random Forest"]["ROC-AUC"], 
              classical_results["Gradient Boosting"]["ROC-AUC"], 
              auc_q]
acc_scores = [classical_results["Support Vector Machine (RBF)"]["Accuracy"], 
              classical_results["Random Forest"]["Accuracy"], 
              classical_results["Gradient Boosting"]["Accuracy"], 
              acc_q]

x = np.arange(len(models_list))
width = 0.25

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 5))

rects1 = ax.bar(x - width, f1_scores, width, label='F1-Score', color='#4A90E2')
rects2 = ax.bar(x, auc_scores, width, label='ROC-AUC', color='#50E3C2')
rects3 = ax.bar(x + width, acc_scores, width, label='Accuracy', color='#F5A623')

ax.set_ylabel('Scores')
ax.set_title('Classical vs. Quantum Models Performance (Credit Card Fraud Detection)')
ax.set_xticks(x)
ax.set_xticklabels(models_list)
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.show()

# Kuantum Kernel sıcaklık haritasını çizdirelim
plt.figure(figsize=(7, 6))
sns.heatmap(train_matrix[:50, :50], cmap='viridis', cbar=True)
plt.title("Quantum Kernel Matrix (First 50x50 samples)")
plt.xlabel("Sample Index")
plt.ylabel("Sample Index")
plt.tight_layout()
plt.show()